# Arabic sentiment data: translating the English set

There's no public Arabic financial sentiment dataset, so this notebook makes one, the way the PRD describes: it machine-translates the English financial sentiment sentences (Financial PhraseBank and FiQA) into Arabic. Each Arabic sentence keeps the label from its English original, so every label still comes from a human annotator; only the wording is generated.

Switch to a GPU first: **Runtime → Change runtime type → T4 GPU**. About 5,000 sentences are left to translate, which should take 10–15 minutes on a T4.

It saves as it goes, so if Colab disconnects, just run it again and it carries on.

When it's done, the training notebook (`train_colab.ipynb`) picks the result up automatically.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('cuda available:', torch.cuda.is_available())

## 2. Find the files on Drive

Everything lives in `My Drive/fintext-raw/translated_sentiment_ar/`:
- `english_sentiment.jsonl`: the English sentences and their labels (the input)
- `rows.jsonl`: the Arabic translations done so far (the output; new ones get added to it)

This is the same folder the training notebook reads from.

In [ ]:
import json, os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
FOLDER = Path('/content/drive/MyDrive/fintext-raw/translated_sentiment_ar')
ENGLISH = FOLDER / 'english_sentiment.jsonl'
OUTPUT = FOLDER / 'rows.jsonl'

if not ENGLISH.exists():
    raise SystemExit(f"Can't find {ENGLISH}. Put english_sentiment.jsonl in that folder and run again.")
print('input: ', ENGLISH)
print('output:', OUTPUT, '(exists)' if OUTPUT.exists() else '(will be created)')

## 3. Install and sign in to HuggingFace

The translation model needs `sentencepiece`. The HuggingFace token is optional; if you've added the `fintext` secret (🔑 in the left sidebar) it gets used, otherwise the download just runs anonymously.

In [ ]:
!pip install -q sentencepiece sacremoses

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('fintext')
    print('HuggingFace token: from the fintext secret')
except Exception:
    print('HuggingFace token: none, downloading anonymously')

## 4. Work out what's left

Reads the English sentences and skips any that are already in `rows.jsonl`.

In [ ]:
from collections import Counter

english = [json.loads(line) for line in ENGLISH.open(encoding='utf-8') if line.strip()]

done = set()
if OUTPUT.exists():
    for line in OUTPUT.open(encoding='utf-8'):
        if line.strip():
            done.add(json.loads(line)['source_text'])

todo = [row for row in english if row['text'] not in done]
print(f'{len(english)} English sentences, {len(done)} already translated, {len(todo)} to go')
print('labels:', dict(Counter(row['sentiment'] for row in english)))

## 5. Translate

Uses `Helsinki-NLP/opus-mt-en-ar`. Sentences are sorted by length first so each batch has similar-length sentences, which is a lot faster on the GPU. Every batch is written to Drive straight away, so nothing is lost if the session drops.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
from tqdm.auto import tqdm

MODEL = 'Helsinki-NLP/opus-mt-en-ar'
BATCH = 64

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = MarianTokenizer.from_pretrained(MODEL)
model = MarianMTModel.from_pretrained(MODEL).to(device).eval()
if device == 'cuda':
    model = model.half()   # half precision, roughly twice as fast on a T4

todo.sort(key=lambda row: len(row['text']))

with OUTPUT.open('a', encoding='utf-8') as out, torch.inference_mode():
    for start in tqdm(range(0, len(todo), BATCH), desc='translating', unit='batch'):
        batch = todo[start:start + BATCH]
        inputs = tokenizer([row['text'] for row in batch], return_tensors='pt',
                           padding=True, truncation=True, max_length=256).to(device)
        generated = model.generate(**inputs, num_beams=4, max_new_tokens=256)
        arabic = tokenizer.batch_decode(generated, skip_special_tokens=True)

        for row, text in zip(batch, arabic):
            out.write(json.dumps({
                'text': text.strip(),
                'language': 'ar',
                'labels': {'sentiment': row['sentiment']},
                'source': row['source'],
                'source_text': row['text'],
                'translator': MODEL,
            }, ensure_ascii=False) + '\n')
        out.flush()

print('done')

## 6. Check the result

Counts, label balance, and a few random examples. I can't read Arabic, so the examples are mainly to catch obvious problems: empty lines, English left untranslated, the same sentence repeated.

In [ ]:
import random, time

rows = [json.loads(line) for line in OUTPUT.open(encoding='utf-8') if line.strip()]
print(f'{len(rows)} translated of {len(english)}')
print('labels:', dict(Counter(r['labels']['sentiment'] for r in rows)))

empty = sum(1 for r in rows if not r['text'])
still_english = sum(1 for r in rows if sum(c.isascii() and c.isalpha() for c in r['text']) > len(r['text']) / 2)
print(f'empty: {empty}   mostly English letters: {still_english}')

for r in random.Random(0).sample(rows, 5):
    print(f"\n[{r['labels']['sentiment']}] {r['source_text'][:100]}")
    print(f"     {r['text'][:100]}")

(FOLDER / 'SOURCE.json').write_text(json.dumps({
    'generated_at': time.strftime('%Y-%m-%d'),
    'translator': MODEL,
    'translated_from': ['financial_phrasebank', 'fiqa_2018'],
    'rows': len(rows),
    'note': 'machine translation of human-labelled English sentences; labels are the English annotators\'',
}, indent=2))

## Next

Run `train_colab.ipynb`. In step 5 it reads `rows.jsonl` from this folder and adds the Arabic sentences to training. Only the ones whose English original is in the training split, so nothing leaks into the test set.